# File 11 — Hybrid Probability-Level Image Branch Fusion
CNN probability + handcrafted tongue features → Logistic Regression → final image-branch score.


In [12]:
import json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.metrics import brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import efficientnet_b0
from torch.utils.data import Dataset, DataLoader
from PIL import Image

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEAT_PATH = Path(r'D:\DIABETES\diabetes_pipeline_outputs\10_hybrid_image_features\10_feature_matrix.csv')
MODEL_COLS_PATH = Path(r'D:\DIABETES\diabetes_pipeline_outputs\10_hybrid_image_features\10_model_feature_columns.json')
QC_COLS_PATH = Path(r'D:\DIABETES\diabetes_pipeline_outputs\10_hybrid_image_features\10_qc_feature_columns.json')
CNN_VAL_PATH = Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\05_lr_validation_predictions_best.csv')
CNN_TEST_PATH = Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation_lighting_robust\06_lr_test_predictions.csv')
CNN_CKPT = Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\best_segmented_lighting_robust_model.pth')
MANIFEST_PATH = Path(r'D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv')
OUTPUT_DIR = Path(r'D:\DIABETES\diabetes_pipeline_outputs\11_hybrid_probability_fusion')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224; BATCH_SIZE = 32; N_BOOTSTRAP = 1000
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

with open(MODEL_COLS_PATH) as f:
    MODEL_FEATURES = json.load(f)
print(f'Model features: {len(MODEL_FEATURES)}')
print(f'Device: {DEVICE}')

def correct_lighting_clahe(image_pil, clip_limit=1.5, tile_grid_size=(8, 8)):
    """Deterministic CLAHE lighting correction. Matches File 10/11 training pipeline."""
    import numpy as np, cv2
    img_rgb = np.array(image_pil.convert('RGB'))
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    rgb_eq = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2RGB)
    return Image.fromarray(rgb_eq)



Model features: 33
Device: cuda


## Load Feature Matrix and CNN Predictions


In [13]:
df_feat = pd.read_csv(FEAT_PATH)
print(f'Feature matrix: {len(df_feat)} rows')

# Load existing CNN predictions
cnn_val = pd.read_csv(CNN_VAL_PATH) if CNN_VAL_PATH.exists() else None
cnn_test = pd.read_csv(CNN_TEST_PATH) if CNN_TEST_PATH.exists() else None

# We need CNN probabilities for ALL splits (train, val, test)
# Val and test may exist. Train needs generation.
df_train_feat = df_feat[df_feat['final_split'] == 'train'].copy()
df_val_feat = df_feat[df_feat['final_split'] == 'val'].copy()
df_test_feat = df_feat[df_feat['final_split'] == 'test'].copy()

print(f'Train: {len(df_train_feat)}, Val: {len(df_val_feat)}, Test: {len(df_test_feat)}')


Feature matrix: 2750 rows
Train: 1930, Val: 408, Test: 412


## Generate CNN Probabilities for All Splits


In [14]:
def pad_square(img):
    w, h = img.size; s = max(w, h)
    new = Image.new('RGB', (s, s), (0,0,0))
    new.paste(img, ((s-w)//2, (s-h)//2))
    return new

# NOTE: CLAHE applied inside SimpleDS.__getitem__ before this transform.
# Transform: pad_square → Resize(224) → ToTensor → Normalize (NO random aug).
cls_transform = T.Compose([
    T.Lambda(pad_square),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

class SimpleDS(Dataset):
    """Dataset applying CLAHE before transform. Matches lighting-robust training pipeline."""
    def __init__(self, paths, transform):
        self.paths = paths; self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        img = correct_lighting_clahe(img)   # CLAHE before classifier transform
        return self.transform(img), idx

def generate_cnn_probs(model, image_paths):
    ds = SimpleDS(image_paths, cls_transform)
    dl = DataLoader(ds, BATCH_SIZE, shuffle=False, num_workers=0)
    probs = np.zeros(len(image_paths))
    model.eval()
    with torch.no_grad():
        for imgs, indices in dl:
            out = model(imgs.to(DEVICE))
            p = torch.sigmoid(out).cpu().numpy().flatten()
            for i, idx_val in enumerate(indices.numpy()):
                probs[idx_val] = p[i]
    return probs

# Load CNN model
cnn_model = efficientnet_b0(weights=None)
cnn_model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(cnn_model.classifier[1].in_features, 1))
ckpt = torch.load(CNN_CKPT, map_location=DEVICE)
if isinstance(ckpt, dict) and 'model_state_dict' in ckpt: cnn_model.load_state_dict(ckpt['model_state_dict'])
elif isinstance(ckpt, dict) and 'state_dict' in ckpt: cnn_model.load_state_dict(ckpt['state_dict'])
else: cnn_model.load_state_dict(ckpt)
cnn_model = cnn_model.to(DEVICE)
cnn_model.eval()

# Generate for all splits
for split_name, split_df in [('train', df_train_feat), ('val', df_val_feat), ('test', df_test_feat)]:
    probs = generate_cnn_probs(cnn_model, split_df['segmented_image_path'].tolist())
    df_feat.loc[df_feat['final_split'] == split_name, 'cnn_probability_diabetes'] = probs
    print(f'{split_name}: CNN probs generated ({len(probs)})')

print(f'CNN probs available for {df_feat["cnn_probability_diabetes"].notna().sum()} images.')


train: CNN probs generated (1930)
val: CNN probs generated (408)
test: CNN probs generated (412)
CNN probs available for 2750 images.


## Prepare Hybrid Input


In [15]:
HYBRID_COLS = ['cnn_probability_diabetes'] + MODEL_FEATURES

# Drop rows with NaN in hybrid columns
df_hybrid = df_feat.dropna(subset=HYBRID_COLS + ['label_binary']).copy()
print(f'Hybrid rows: {len(df_hybrid)}')

train = df_hybrid[df_hybrid['final_split'] == 'train']
val   = df_hybrid[df_hybrid['final_split'] == 'val']
test  = df_hybrid[df_hybrid['final_split'] == 'test']

X_train = train[HYBRID_COLS].values; y_train = train['label_binary'].values
X_val   = val[HYBRID_COLS].values;   y_val   = val['label_binary'].values
X_test  = test[HYBRID_COLS].values;  y_test  = test['label_binary'].values

# Scale (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

joblib.dump(scaler, OUTPUT_DIR / '11_hybrid_scaler.joblib')
print(f'Scaler fit on train ({len(X_train)} samples). Saved.')

pd.DataFrame([{'split':'train','count':len(train),'diabetes':int(y_train.sum())},
              {'split':'val','count':len(val),'diabetes':int(y_val.sum())},
              {'split':'test','count':len(test),'diabetes':int(y_test.sum())}]
).to_csv(OUTPUT_DIR / '11_hybrid_train_val_test_summary.csv', index=False)
pd.DataFrame({'feature': HYBRID_COLS}).to_csv(OUTPUT_DIR / '11_hybrid_selected_features.csv', index=False)


Hybrid rows: 2750
Scaler fit on train (1930 samples). Saved.


## Train Hybrid Logistic Regression


In [16]:
def compute_metrics(yt, yp, t=0.5):
    yd = (yp >= t).astype(int)
    tn,fp,fn,tp = confusion_matrix(yt,yd,labels=[0,1]).ravel()
    r=tp/(tp+fn) if tp+fn>0 else 0; sp=tn/(tn+fp) if tn+fp>0 else 0
    ppv=tp/(tp+fp) if tp+fp>0 else 0; npv=tn/(tn+fn) if tn+fn>0 else 0
    f1=2*ppv*r/(ppv+r) if ppv+r>0 else 0
    return {'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
            'diabetes_recall':r,'fnr':fn/(tp+fn) if tp+fn>0 else 0,
            'specificity':sp,'ppv':ppv,'npv':npv,'f1':f1,
            'balanced_accuracy':(r+sp)/2,'accuracy':(tp+tn)/(tp+tn+fp+fn),
            'brier':brier_score_loss(yt,yp)}

# Tune C on validation
best_C, best_score, best_threshold = 1.0, -1, 0.5
val_results = []

for C in [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    lr = LogisticRegression(C=C, penalty='l2', solver='lbfgs', max_iter=2000, random_state=SEED)
    lr.fit(X_train_s, y_train)
    yp_val = lr.predict_proba(X_val_s)[:, 1]
    roc = roc_auc_score(y_val, yp_val)
    pr = average_precision_score(y_val, yp_val)
    
    for t in np.arange(0.05, 0.96, 0.05):
        m = compute_metrics(y_val, yp_val, t)
        score = 0.45*m['diabetes_recall'] + 0.25*m['specificity'] + 0.20*pr + 0.10*m['balanced_accuracy']
        val_results.append({'C':C,'threshold':round(t,2),'screening_score':score,'recall':m['diabetes_recall'],'spec':m['specificity'],'roc_auc':roc,'pr_auc':pr})
        if score > best_score:
            best_score, best_C, best_threshold = score, C, round(t, 2)

pd.DataFrame(val_results).to_csv(OUTPUT_DIR / '11_hybrid_validation_metrics_by_threshold.csv', index=False)
print(f'Best C={best_C}, threshold={best_threshold}, score={best_score:.4f}')


Best C=1.0, threshold=0.3, score=0.9350


In [17]:
# Fit final model with best C
final_model = LogisticRegression(C=best_C, penalty='l2', solver='lbfgs', max_iter=2000, random_state=SEED)
final_model.fit(X_train_s, y_train)
joblib.dump(final_model, OUTPUT_DIR / '11_hybrid_model.joblib')
print('Final model saved.')

# Feature coefficients
coefs = pd.DataFrame({'feature': HYBRID_COLS, 'coefficient': final_model.coef_[0]}).sort_values('coefficient', key=abs, ascending=False)
coefs.to_csv(OUTPUT_DIR / '11_hybrid_feature_coefficients.csv', index=False)
print('Top features by |coefficient|:')
print(coefs.head(10).to_string(index=False))

# Best model summary
with open(OUTPUT_DIR / '11_hybrid_best_model_summary.json', 'w') as f:
    json.dump({'C': best_C, 'best_threshold': best_threshold, 'best_screening_score': best_score,
               'n_features': len(HYBRID_COLS), 'model_type': 'LogisticRegression_L2'}, f, indent=2)

# Validation predictions
yp_val = final_model.predict_proba(X_val_s)[:, 1]
pd.DataFrame({'y_true': y_val, 'y_prob': yp_val}).to_csv(OUTPUT_DIR / '11_hybrid_validation_predictions.csv', index=False)


Final model saved.
Top features by |coefficient|:
                 feature  coefficient
cnn_probability_diabetes     3.693234
              hsv_mean_s     1.536123
         saturation_mean     1.536123
              hsv_mean_h    -1.438052
     white_coating_ratio     1.372142
               rgb_std_g    -1.356340
              rgb_mean_b     1.049697
              rgb_mean_r     1.025922
              lab_mean_b    -0.990435
              hsv_mean_v    -0.902137


## Locked Test Evaluation


In [18]:
yp_test = final_model.predict_proba(X_test_s)[:, 1]
pd.DataFrame({'y_true': y_test, 'y_prob': yp_test}).to_csv(OUTPUT_DIR / '11_hybrid_test_predictions.csv', index=False)

m05 = compute_metrics(y_test, yp_test, 0.5)
msel = compute_metrics(y_test, yp_test, best_threshold)
roc_auc = roc_auc_score(y_test, yp_test)
pr_auc = average_precision_score(y_test, yp_test)

pd.DataFrame([m05]).to_csv(OUTPUT_DIR / '11_hybrid_test_metrics_threshold_0_5.csv', index=False)
pd.DataFrame([msel]).to_csv(OUTPUT_DIR / '11_hybrid_test_metrics_selected_threshold.csv', index=False)
pd.DataFrame([{'roc_auc':roc_auc,'pr_auc':pr_auc}]).to_csv(OUTPUT_DIR / '11_hybrid_test_probability_metrics.csv', index=False)

print(f'Test ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}')
print(f'Test recall@{best_threshold}: {msel["diabetes_recall"]:.4f}, spec: {msel["specificity"]:.4f}')


Test ROC AUC: 0.9702, PR AUC: 0.9699
Test recall@0.3: 0.9581, spec: 0.8680


## Bootstrap CIs


In [19]:
rng = np.random.default_rng(SEED)
ci_rows = []
for name, fn in [
    ('recall', lambda yt,yp: compute_metrics(yt,yp,best_threshold)['diabetes_recall']),
    ('fnr', lambda yt,yp: compute_metrics(yt,yp,best_threshold)['fnr']),
    ('specificity', lambda yt,yp: compute_metrics(yt,yp,best_threshold)['specificity']),
    ('ppv', lambda yt,yp: compute_metrics(yt,yp,best_threshold)['ppv']),
    ('npv', lambda yt,yp: compute_metrics(yt,yp,best_threshold)['npv']),
    ('f1', lambda yt,yp: compute_metrics(yt,yp,best_threshold)['f1']),
    ('balanced_accuracy', lambda yt,yp: compute_metrics(yt,yp,best_threshold)['balanced_accuracy']),
    ('roc_auc', lambda yt,yp: roc_auc_score(yt,yp)),
    ('pr_auc', lambda yt,yp: average_precision_score(yt,yp)),
]:
    vals = [fn(y_test[idx:=rng.choice(len(y_test),len(y_test),replace=True)], yp_test[idx]) for _ in range(N_BOOTSTRAP)]
    ci_rows.append({'metric':name,'ci_2.5':np.percentile(vals,2.5),'ci_97.5':np.percentile(vals,97.5)})
pd.DataFrame(ci_rows).to_csv(OUTPUT_DIR / '11_hybrid_bootstrap_confidence_intervals.csv', index=False)
print('Bootstrap CIs saved.')


Bootstrap CIs saved.


## Plots


In [20]:
# Confusion matrix
cm = confusion_matrix(y_test, (yp_test>=best_threshold).astype(int), labels=[0,1])
fig,ax=plt.subplots(figsize=(6,6))
ax.matshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=14)
ax.set_xticklabels(['','ND','D']); ax.set_yticklabels(['','ND','D'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Hybrid LR — threshold {best_threshold}')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'11_hybrid_confusion_matrix_selected_threshold.png',dpi=100); plt.close()

# ROC
fpr,tpr,_=roc_curve(y_test,yp_test)
fig,ax=plt.subplots(figsize=(7,7)); ax.plot(fpr,tpr,label=f'AUC={roc_auc:.3f}'); ax.plot([0,1],[0,1],'k--'); ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'11_hybrid_roc_curve.png',dpi=100); plt.close()

# PR
prec,rec,_=precision_recall_curve(y_test,yp_test)
fig,ax=plt.subplots(figsize=(7,7)); ax.plot(rec,prec,label=f'AUC={pr_auc:.3f}'); ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'11_hybrid_pr_curve.png',dpi=100); plt.close()

# Calibration
fp_cal,mp_cal=calibration_curve(y_test,yp_test,n_bins=10,strategy='uniform')
fig,ax=plt.subplots(figsize=(7,7)); ax.plot(mp_cal,fp_cal,'s-'); ax.plot([0,1],[0,1],'k--')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'11_hybrid_calibration_curve.png',dpi=100); plt.close()

print('Plots saved.')


Plots saved.


## Handoff


In [21]:
handoff = f"""FILE 11 HANDOFF — HYBRID PROBABILITY FUSION
Status: PASS
Model: Logistic Regression L2, C={best_C}
Features: CNN probability + {len(MODEL_FEATURES)} handcrafted tongue features
CNN base: lighting-robust CNN (best_segmented_lighting_robust_model.pth)
CLAHE applied before CNN probability generation: YES
CLAHE-based handcrafted features used: YES
QC features excluded: YES
Scaler fit on train only: YES
Threshold selected on validation only: YES
Test threshold tuning: NO
Scaler: fit on train only
Threshold: {best_threshold} (selected on val)

Test metrics at selected threshold {best_threshold}:
  Recall: {msel['diabetes_recall']:.4f}, FNR: {msel['fnr']:.4f}
  Specificity: {msel['specificity']:.4f}
  PPV: {msel['ppv']:.4f}, NPV: {msel['npv']:.4f}
  F1: {msel['f1']:.4f}, Balanced Acc: {msel['balanced_accuracy']:.4f}
  Accuracy: {msel['accuracy']:.4f}, Brier: {msel['brier']:.4f}
  ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}

No threshold tuning on test.
Proceed to File 13 for comparison with CNN-only.
"""
with open(OUTPUT_DIR / '11_hybrid_handoff_summary.txt', 'w') as f:
    f.write(handoff)
print(handoff)


FILE 11 HANDOFF — HYBRID PROBABILITY FUSION
Status: PASS
Model: Logistic Regression L2, C=1.0
Features: CNN probability + 33 handcrafted tongue features
CNN base: lighting-robust CNN (best_segmented_lighting_robust_model.pth)
CLAHE applied before CNN probability generation: YES
CLAHE-based handcrafted features used: YES
QC features excluded: YES
Scaler fit on train only: YES
Threshold selected on validation only: YES
Test threshold tuning: NO
Scaler: fit on train only
Threshold: 0.3 (selected on val)

Test metrics at selected threshold 0.3:
  Recall: 0.9581, FNR: 0.0419
  Specificity: 0.8680
  PPV: 0.8879, NPV: 0.9500
  F1: 0.9217, Balanced Acc: 0.9131
  Accuracy: 0.9150, Brier: 0.0621
  ROC AUC: 0.9702, PR AUC: 0.9699

No threshold tuning on test.
Proceed to File 13 for comparison with CNN-only.

